# 30 — Block 3: Finetune trên kết quả proposal + đánh giá

Nhận các `export.csv` (từ notebook 20, P-026 hoặc nơi khác) → chốt manifest → mở nhãn BDD (simulated annotation) → finetune từ base model
→ đánh giá trên final test. So với **model base**, **`random` cùng seed / K** (nhánh tham chiếu) và **REPLAY_ONLY** (train thêm mà không có ảnh mới).
Harness không chứa logic chọn mẫu.

**Input:** output notebook 10 + 20 (hoặc dataset chứa `art/proposals/`). **GPU:** cần. **Output:** `art/runs/`, `art/eval/report.md`.

**Hết giờ session / chạy tiếp:** Save Version → gắn output version vừa xong làm input của chính notebook này → chạy lại.
Mọi bước đã có kết quả được bỏ qua (cache).

In [ ]:
# === Setup (giống nhau ở mọi notebook) ===
SMOKE = True   # True: pipeline thật trên tập nhỏ. Run thật: False
SCENARIO = "missing"   # missing: base chưa có novel class | weak: base có ít ảnh novel
REPO_URL = "https://github.com/Le-Anh-Duy/topicx-yolo-expansion.git"
COMMIT = "main"   # run thật: pin commit hash
import os, subprocess, sys
R = "/tmp/repo"
if not os.path.exists(R):
    subprocess.run(["git", "clone", "-q", REPO_URL, R], check=True)
# luôn lấy bản mới nhất của COMMIT từ GitHub (kể cả khi /tmp/repo còn từ lần chạy trước trong session tương tác)
subprocess.run(["git", "-C", R, "fetch", "-q", "origin", COMMIT], check=True)
subprocess.run(["git", "-C", R, "checkout", "-q", "-f", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{R}/requirements.txt"], check=True)
for p in (f"{R}/src", R):   # `topicx` (harness) và `src.rav` (pipeline proposal, cùng đường import với P-026)
    if p not in sys.path:
        sys.path.insert(0, p)
for m in [m for m in sys.modules if m.split(".")[0] in ("topicx", "src")]:
    del sys.modules[m]   # chạy lại cell thì nạp lại code mới
import numpy as np, pandas as pd
from topicx import common as C, pipeline as P, proposals as PR
cfg = C.load_config(SMOKE, SCENARIO)
commit = subprocess.run(["git", "-C", R, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("code:", commit)
print(f"kịch bản={SCENARIO} | artifact={cfg['art']} | copy từ input: {C.sync_inputs(cfg)} file")

In [ ]:
STAGE = "finetune"
# Test (CPU, vài giây): protocol của harness + test lõi P-026 + component mới. Lỗi thì dừng notebook.
subprocess.run([sys.executable, "-m", "pytest", "-q", f"{R}/tests"], check=True)
C.env_report(cfg, STAGE);

## Health check dataset
Vài giây, không đọc cả file nhãn. Có mục FAIL thì notebook dừng ở đây.

In [ ]:
hc, sample = P.health_check(cfg)
if sample is not None:
    display(sample.resize((640, 360)))   # box đỏ phải ôm đúng vật thể

In [ ]:
NOVEL, REASON = "bus", "EDA / giả thuyết"
pcfg = PR.load_proposal_config(cfg)
SEEDS, KS = cfg["seeds"], pcfg["ks"]
REFERENCE = ["REPLAY_ONLY"]        # thêm "ORACLE_POSITIVE" nếu muốn mốc dùng nhãn ẩn

## Base model

In [ ]:
meta = P.prepare_splits(cfg, NOVEL, REASON)   # dùng lại splits của notebook 10
W, bm = P.train_base(cfg)
display(pd.DataFrame(bm["dev_ap"]).T)

## Nhận kết quả proposal → manifest đã chốt → thống kê tập được chọn

In [ ]:
imp = P.import_exports(cfg, seeds=SEEDS, ks=KS)
display(imp)
if "ORACLE_POSITIVE" in REFERENCE:
    P.oracle_positive(cfg, SEEDS, KS)
BRANCHES = sorted(imp.branch.unique()) + REFERENCE
print("nhánh:", BRANCHES, f"| ước tính train: {P.estimate_hours(cfg, len(BRANCHES), SEEDS, KS):.1f} h GPU")
sel = P.selection_eval(cfg)
display(sel.groupby(["k", "branch"])[[c for c in ("precision_at_k", "novel_instances", "base_instances", "nn_cos_mean", "shortfall") if c in sel]].mean())

## Finetune từng nhánh

In [ ]:
todo = P.train_branches(cfg, BRANCHES, SEEDS, KS)
print("còn thiếu (chạy tiếp ở version sau):", todo or "không")

## Đánh giá trên final test

In [ ]:
res = P.evaluate(cfg, BRANCHES, SEEDS, KS)
print(f"base model: base mAP50-95 {res['base_map']:.4f}, AP novel {res['base_novel_ap']:.4f} | tham chiếu: {res['ref']} | thiếu: {res['missing'] or 'không'}")
print("Uplift AP50-95 novel so với tham chiếu, từng seed:"); display(res["uplift"])
print("AP50-95 từng class (0–100):"); display(res["per_class"])
display(res["summary"])

## Gói kết quả
Zip file json/csv/md nhỏ (không có weights, embedding, nhãn oracle) → tab Output.

In [ ]:
P.bundle(cfg, STAGE)